# 07. 장애인콜택시 예상 대기시간 HistGradientBoostingRegressor 모델링

이 노트북은 05번, 06번 노트북에서 만든 장애인콜택시 바로콜 대기시간 모델링 흐름을 바탕으로 `HistGradientBoostingRegressor` 모델을 학습하고 평가한다.

예측 대상은 바로콜 기준 `접수→승차 대기시간`이다.

```text
예상 대기시간 = 접수→승차 대기시간
총 예상 이동시간 = 예상 대기시간 + 차량 이동시간
```

차량 이동시간은 지도/경로 API에서 별도로 가져오고, 이 모델은 호출 후 실제 승차까지 걸리는 대기시간만 예측한다.

## 모델 범위

이번 노트북에서는 바로콜 승차완료 건만 사용한다.

```text
임차택시_바로콜
특장차_바로콜
```

전일접수와 심야시간 사전예약은 호출 구조가 다르므로 이번 HGB 모델에서는 제외한다.


## 1. 라이브러리 및 경로 설정

로컬 VSCode와 Google Colab 양쪽에서 실행할 수 있도록 경로를 자동 설정한다.

필요한 CSV 파일은 다음 2개다.

```text
임차택시_대기시간_전처리.csv
특장차_대기시간_전처리_접수유형분류.csv
```


In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount("/content/drive")
except ImportError:
    IN_COLAB = False

import json
import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    precision_score,
    r2_score,
    recall_score,
)
from sklearn.model_selection import ParameterGrid, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

RANDOM_STATE = 42

if IN_COLAB:
    DRIVE_ROOT = Path("/content/drive/MyDrive")
    PROJECT_ROOT = DRIVE_ROOT / "calltaxi-DA"
    DATA_DIR = PROJECT_ROOT / "data" / "processed"
    if not DATA_DIR.exists():
        DATA_DIR = DRIVE_ROOT
else:
    cwd = Path.cwd()
    PROJECT_ROOT = cwd.parent if cwd.name == "notebooks_waiting_time" else cwd
    DATA_DIR = PROJECT_ROOT / "data" / "processed"

MODEL_DIR = PROJECT_ROOT / "models" / "waiting_time"
REPORT_DIR = PROJECT_ROOT / "reports" / "waiting_time_model"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

RENTAL_PATH = DATA_DIR / "임차택시_대기시간_전처리.csv"
SPECIAL_PATH = DATA_DIR / "특장차_대기시간_전처리_접수유형분류.csv"

if not RENTAL_PATH.exists():
    raise FileNotFoundError(f"임차택시 CSV를 찾을 수 없습니다: {RENTAL_PATH}")

if not SPECIAL_PATH.exists():
    raise FileNotFoundError(f"특장차 CSV를 찾을 수 없습니다: {SPECIAL_PATH}")

print("IN_COLAB:", IN_COLAB)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("RENTAL_PATH:", RENTAL_PATH)
print("SPECIAL_PATH:", SPECIAL_PATH)


## 2. 모델링 데이터셋 생성

임차택시와 특장차 바로콜 데이터를 하나로 합친다.

`model_group` 컬럼으로 차량·호출유형을 구분한다.

```text
임차택시_바로콜
특장차_바로콜
```

최종 예측 대상은 `target_min = 접수_승차_분`이다.


In [ ]:
def load_modeling_dataset(extreme_wait_threshold=180.0):
    rental_cols = [
        "접수일시", "승차일시",
        "출발구", "출발동", "목적구", "목적동", "이용목적",
        "접수_배차_분", "배차_승차_분", "접수_승차_분",
        "임차택시_바로콜여부", "대기시간분석_포함여부",
    ]

    special_cols = [
        "접수일시", "승차일시",
        "출발구", "출발동", "목적구", "목적동", "이용목적",
        "접수_배차_분", "배차_승차_분", "접수_승차_분",
        "특장차_바로콜_후보여부", "특장차_접수유형_후보_최종",
    ]

    rental = pd.read_csv(RENTAL_PATH, usecols=rental_cols, low_memory=False)
    rental = rental[
        rental["임차택시_바로콜여부"].fillna(False).astype(bool)
        & rental["대기시간분석_포함여부"].fillna(False).astype(bool)
    ].copy()
    rental["model_group"] = "임차택시_바로콜"

    special = pd.read_csv(SPECIAL_PATH, usecols=special_cols, low_memory=False)
    special = special[
        special["특장차_바로콜_후보여부"].fillna(False).astype(bool)
        | special["특장차_접수유형_후보_최종"].eq("바로콜 후보")
    ].copy()
    special["model_group"] = "특장차_바로콜"

    keep_cols = [
        "model_group",
        "접수일시", "승차일시",
        "출발구", "출발동", "목적구", "목적동", "이용목적",
        "접수_배차_분", "배차_승차_분", "접수_승차_분",
    ]

    data = pd.concat([rental[keep_cols], special[keep_cols]], ignore_index=True)

    data["접수일시"] = pd.to_datetime(data["접수일시"], errors="coerce")
    data["승차일시"] = pd.to_datetime(data["승차일시"], errors="coerce")

    for col in ["접수_배차_분", "배차_승차_분", "접수_승차_분"]:
        data[col] = pd.to_numeric(data[col], errors="coerce")

    data = data[
        data["접수일시"].notna()
        & data["승차일시"].notna()
        & data["접수_승차_분"].notna()
        & data["접수_승차_분"].ge(0)
    ].copy()

    data["target_min"] = data["접수_승차_분"]
    data["is_extreme_wait"] = data["target_min"].gt(extreme_wait_threshold).astype("int8")

    data["hour"] = data["접수일시"].dt.hour.astype("int16")
    data["dayofweek"] = data["접수일시"].dt.dayofweek.astype("int16")
    data["month"] = data["접수일시"].dt.month.astype("int16")
    data["is_weekend"] = data["dayofweek"].isin([5, 6]).astype("int8")
    data["is_night"] = data["hour"].between(0, 6, inclusive="both").astype("int8")
    data["is_commute"] = (
        data["hour"].between(7, 9, inclusive="both")
        | data["hour"].between(17, 19, inclusive="both")
    ).astype("int8")

    for col in ["출발구", "출발동", "목적구", "목적동", "이용목적"]:
        data[col] = data[col].fillna("미상").astype(str)

    return data


data = load_modeling_dataset()
data.shape


In [ ]:
display(data[[
    "model_group", "접수일시", "출발구", "목적구", "이용목적",
    "target_min", "hour", "dayofweek", "month", "is_extreme_wait",
]].head())

display(
    data.groupby("model_group")
    .agg(
        rows=("target_min", "size"),
        median=("target_min", "median"),
        p90=("target_min", lambda x: x.quantile(0.90)),
        mean=("target_min", "mean"),
    )
    .round(2)
)


## 3. 세부이동유형 생성

출발구와 목적구를 기준으로 이동유형을 구분한다.

```text
구 내 이동
구 간 이동
서울→서울 외
서울 외→서울
서울 외↔서울 외
```

06번 노트북에서 세부이동유형을 추가했을 때 성능이 소폭 개선되었으므로 HGB 모델에도 포함한다.


In [ ]:
SEOUL_GU = {
    "강남구", "강동구", "강북구", "강서구", "관악구",
    "광진구", "구로구", "금천구", "노원구", "도봉구",
    "동대문구", "동작구", "마포구", "서대문구", "서초구",
    "성동구", "성북구", "송파구", "양천구", "영등포구",
    "용산구", "은평구", "종로구", "중구", "중랑구",
}


def classify_move_type(row):
    origin = row["출발구"]
    dest = row["목적구"]

    origin_in_seoul = origin in SEOUL_GU
    dest_in_seoul = dest in SEOUL_GU

    if origin_in_seoul and dest_in_seoul:
        if origin == dest:
            return "구 내 이동"
        return "구 간 이동"

    if origin_in_seoul and not dest_in_seoul:
        return "서울→서울 외"

    if not origin_in_seoul and dest_in_seoul:
        return "서울 외→서울"

    return "서울 외↔서울 외"


data["세부이동유형"] = data.apply(classify_move_type, axis=1)

move_type_summary = (
    data.groupby(["model_group", "세부이동유형"])
    .size()
    .reset_index(name="건수")
)
move_type_summary["그룹내비율(%)"] = (
    move_type_summary["건수"]
    / move_type_summary.groupby("model_group")["건수"].transform("sum")
    * 100
)

display(move_type_summary.round(2))


## 4. 수요 proxy 피처 생성

실제 예측 시점에서 현재 대기열 길이, 운행 중 차량 수, 배차 가능 차량 수를 알 수 없기 때문에 현재 데이터에서 만들 수 있는 수요 proxy를 사용한다.

사용하는 proxy는 다음과 같다.

```text
request_count_prev_30m
request_count_prev_60m
model_group_request_count_prev_60m
origin_gu_request_count_prev_60m
```

rolling count는 반드시 현재 접수시각 이전 데이터만 사용해야 하므로 `closed="left"`를 사용한다.


In [ ]:
def add_global_previous_request_counts(
    frame,
    time_col="접수일시",
    windows=("30min", "60min"),
):
    result = frame.copy()
    result["_original_index"] = np.arange(len(result))
    result = result.sort_values(time_col).copy()
    result["_request_count"] = 1
    temp = result.set_index(time_col)

    for window in windows:
        col_name = f"request_count_prev_{window.replace('min', 'm')}"
        result[col_name] = (
            temp["_request_count"]
            .rolling(window=window, closed="left")
            .sum()
            .fillna(0)
            .to_numpy()
        )

    result = result.sort_values("_original_index").drop(
        columns=["_original_index", "_request_count"]
    )
    return result


def add_group_previous_request_counts(
    frame,
    group_col="model_group",
    time_col="접수일시",
    window="60min",
):
    result = frame.copy()
    result["_original_index"] = np.arange(len(result))
    count_col = f"{group_col}_request_count_prev_{window.replace('min', 'm')}"
    pieces = []

    for _, group_df in result.groupby(group_col):
        group_df = group_df.sort_values(time_col).copy()
        group_df["_request_count"] = 1
        temp = group_df.set_index(time_col)
        group_df[count_col] = (
            temp["_request_count"]
            .rolling(window=window, closed="left")
            .sum()
            .fillna(0)
            .to_numpy()
        )
        pieces.append(group_df.drop(columns=["_request_count"]))

    result = (
        pd.concat(pieces, ignore_index=True)
        .sort_values("_original_index")
        .drop(columns=["_original_index"])
    )
    return result


def add_origin_gu_previous_request_counts(
    frame,
    origin_col="출발구",
    time_col="접수일시",
    window="60min",
):
    result = frame.copy()
    result["_original_index"] = np.arange(len(result))
    count_col = f"origin_gu_request_count_prev_{window.replace('min', 'm')}"
    pieces = []

    for _, group_df in result.groupby(origin_col):
        group_df = group_df.sort_values(time_col).copy()
        group_df["_request_count"] = 1
        temp = group_df.set_index(time_col)
        group_df[count_col] = (
            temp["_request_count"]
            .rolling(window=window, closed="left")
            .sum()
            .fillna(0)
            .to_numpy()
        )
        pieces.append(group_df.drop(columns=["_request_count"]))

    result = (
        pd.concat(pieces, ignore_index=True)
        .sort_values("_original_index")
        .drop(columns=["_original_index"])
    )
    return result


data = add_global_previous_request_counts(data)
data = add_group_previous_request_counts(data)
data = add_origin_gu_previous_request_counts(data)

display(data[[
    "접수일시", "model_group", "출발구", "목적구",
    "request_count_prev_30m", "request_count_prev_60m",
    "model_group_request_count_prev_60m",
    "origin_gu_request_count_prev_60m",
    "target_min",
]].head())


## 5. Train / Validation / Test 분리

`month × model_group` 기준으로 층화하여 70%, 15%, 15%로 나눈다.

시계열 예측 모델이 아니라 네비게이터용 일반화 모델이므로 특정 월이나 차량유형이 한 split에 몰리지 않도록 한다.


In [ ]:
def split_train_valid_test(data):
    data = data.copy()
    data["split_strata"] = data["month"].astype(str) + "_" + data["model_group"].astype(str)

    train, temp = train_test_split(
        data,
        test_size=0.30,
        random_state=RANDOM_STATE,
        stratify=data["split_strata"],
    )

    valid, test = train_test_split(
        temp,
        test_size=0.50,
        random_state=RANDOM_STATE,
        stratify=temp["split_strata"],
    )

    train = train.drop(columns=["split_strata"]).reset_index(drop=True)
    valid = valid.drop(columns=["split_strata"]).reset_index(drop=True)
    test = test.drop(columns=["split_strata"]).reset_index(drop=True)

    return train, valid, test


train, valid, test = split_train_valid_test(data)
train.shape, valid.shape, test.shape


## 6. 장시간 대기율 proxy 피처 생성

장시간 대기 기준은 train set의 `접수→승차` 90분위수로 잡는다.

```text
장시간 대기 = 접수→승차 대기시간 >= train set 90분위수
```

`origin_gu_hour_long_wait_rate`, `model_group_hour_long_wait_rate`는 target에서 파생된 집계 피처이므로 leakage를 조심해야 한다.

- validation/test에는 train set에서 계산한 장시간 대기율만 붙인다.
- train에는 out-of-fold 방식으로 장시간 대기율을 붙인다.


In [ ]:
long_wait_threshold = train["target_min"].quantile(0.90)
threshold_by_group = train.groupby("model_group")["target_min"].quantile(0.90).to_dict()

long_wait_threshold, threshold_by_group


In [ ]:
def make_long_wait_rate_tables(train_frame, threshold):
    tmp = train_frame.copy()
    tmp["_long_wait"] = tmp["target_min"].ge(threshold).astype(int)

    global_rate = tmp["_long_wait"].mean()

    origin_gu_hour = tmp.groupby(["출발구", "hour"])["_long_wait"].mean().to_dict()
    origin_gu = tmp.groupby("출발구")["_long_wait"].mean().to_dict()

    model_group_hour = tmp.groupby(["model_group", "hour"])["_long_wait"].mean().to_dict()
    model_group = tmp.groupby("model_group")["_long_wait"].mean().to_dict()

    return {
        "global_rate": global_rate,
        "origin_gu_hour": origin_gu_hour,
        "origin_gu": origin_gu,
        "model_group_hour": model_group_hour,
        "model_group": model_group,
    }


def apply_long_wait_rate_tables(frame, tables):
    result = frame.copy()
    global_rate = tables["global_rate"]

    result["origin_gu_hour_long_wait_rate"] = [
        tables["origin_gu_hour"].get(
            (r.출발구, int(r.hour)),
            tables["origin_gu"].get(r.출발구, global_rate),
        )
        for r in result[["출발구", "hour"]].itertuples(index=False)
    ]

    result["model_group_hour_long_wait_rate"] = [
        tables["model_group_hour"].get(
            (r.model_group, int(r.hour)),
            tables["model_group"].get(r.model_group, global_rate),
        )
        for r in result[["model_group", "hour"]].itertuples(index=False)
    ]

    return result


def add_oof_long_wait_rate_features(train, valid, test, threshold, n_splits=5):
    train_result = train.copy()
    valid_result = valid.copy()
    test_result = test.copy()

    train_result["origin_gu_hour_long_wait_rate"] = np.nan
    train_result["model_group_hour_long_wait_rate"] = np.nan

    strata = train_result["month"].astype(str) + "_" + train_result["model_group"].astype(str)

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    for fold, (fit_idx, oof_idx) in enumerate(skf.split(train_result, strata), start=1):
        fit_frame = train_result.iloc[fit_idx].copy()
        oof_frame = train_result.iloc[oof_idx].copy()

        tables = make_long_wait_rate_tables(fit_frame, threshold)
        oof_with_rate = apply_long_wait_rate_tables(oof_frame, tables)

        train_result.loc[oof_idx, "origin_gu_hour_long_wait_rate"] = (
            oof_with_rate["origin_gu_hour_long_wait_rate"].to_numpy()
        )
        train_result.loc[oof_idx, "model_group_hour_long_wait_rate"] = (
            oof_with_rate["model_group_hour_long_wait_rate"].to_numpy()
        )

        print(f"OOF fold {fold}/{n_splits} 완료")

    full_tables = make_long_wait_rate_tables(train_result, threshold)
    valid_result = apply_long_wait_rate_tables(valid_result, full_tables)
    test_result = apply_long_wait_rate_tables(test_result, full_tables)

    return train_result, valid_result, test_result, full_tables


train, valid, test, long_wait_rate_tables = add_oof_long_wait_rate_features(
    train,
    valid,
    test,
    long_wait_threshold,
    n_splits=5,
)


## 7. 평가 함수 정의

회귀 모델이므로 기본 성능은 MAE, Median AE, RMSE, R²로 본다.

네비게이터에서는 장시간 대기 위험 안내도 중요하므로 예측값이 기준 시간 이상인지에 대해 Precision, Recall, F1도 함께 본다.


In [ ]:
def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "Median_AE": median_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }


def risk_metrics(y_true, y_pred, threshold):
    actual = y_true >= threshold
    pred = y_pred >= threshold

    return {
        "threshold": threshold,
        "Accuracy": accuracy_score(actual, pred),
        "Precision": precision_score(actual, pred, zero_division=0),
        "Recall": recall_score(actual, pred, zero_division=0),
        "F1": f1_score(actual, pred, zero_division=0),
        "Confusion_Matrix": confusion_matrix(actual, pred, labels=[False, True]).tolist(),
    }


def evaluate_predictions(frame, pred, threshold):
    y = frame["target_min"].to_numpy()
    return {
        **regression_metrics(y, pred),
        **{f"risk_{k}": v for k, v in risk_metrics(y, pred, threshold).items()},
    }


def evaluate_by_group(frame, pred, threshold_by_group):
    tmp = frame[["model_group", "target_min"]].copy()
    tmp["prediction"] = pred

    rows = []

    for group, group_df in tmp.groupby("model_group"):
        y = group_df["target_min"].to_numpy()
        p = group_df["prediction"].to_numpy()
        threshold = threshold_by_group[group]

        rows.append({
            "model_group": group,
            "rows": len(group_df),
            **regression_metrics(y, p),
            **{
                f"risk_{k}": v
                for k, v in risk_metrics(y, p, threshold).items()
                if k != "Confusion_Matrix"
            },
        })

    return pd.DataFrame(rows)


## 8. HGB 모델 입력 변수

06번 노트북에서 성능 개선에 사용한 최종 피처 세트를 동일하게 사용한다.

```text
model_group
출발구
목적구
세부이동유형
hour
dayofweek
month
is_weekend
is_night
is_commute
request_count_prev_30m
request_count_prev_60m
model_group_request_count_prev_60m
origin_gu_request_count_prev_60m
origin_gu_hour_long_wait_rate
model_group_hour_long_wait_rate
```

출발동/목적동은 고유값이 많아 1차 HGB 모델에서는 제외한다.


In [ ]:
HGB_FEATURES = [
    "model_group",
    "출발구",
    "목적구",
    "세부이동유형",
    "hour",
    "dayofweek",
    "month",
    "is_weekend",
    "is_night",
    "is_commute",
    "request_count_prev_30m",
    "request_count_prev_60m",
    "model_group_request_count_prev_60m",
    "origin_gu_request_count_prev_60m",
    "origin_gu_hour_long_wait_rate",
    "model_group_hour_long_wait_rate",
]

CATEGORICAL_COLS = [
    "model_group",
    "출발구",
    "목적구",
    "세부이동유형",
]

NUMERIC_COLS = [
    col for col in HGB_FEATURES
    if col not in CATEGORICAL_COLS
]

train[HGB_FEATURES].head()


## 9. HistGradientBoostingRegressor 모델 함수

범주형 변수는 `OrdinalEncoder`로 숫자화한다.

`HistGradientBoostingRegressor`는 트리 기반 모델이므로 선형 모델처럼 숫자의 크기 자체를 계수로 해석하지 않는다. 다만 범주형 변수 처리를 위해 categorical feature index를 모델에 넘긴다.


In [ ]:
def build_hgb_pipeline(params=None):
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "cat",
                OrdinalEncoder(
                    handle_unknown="use_encoded_value",
                    unknown_value=np.nan,
                    encoded_missing_value=np.nan,
                ),
                CATEGORICAL_COLS,
            ),
            (
                "num",
                "passthrough",
                NUMERIC_COLS,
            ),
        ],
        verbose_feature_names_out=False,
    )

    default_params = {
        "loss": "squared_error",
        "learning_rate": 0.05,
        "max_iter": 300,
        "max_leaf_nodes": 31,
        "min_samples_leaf": 50,
        "l2_regularization": 0.1,
        "early_stopping": True,
        "n_iter_no_change": 20,
        "validation_fraction": 0.1,
        "random_state": RANDOM_STATE,
        "verbose": 1,
    }

    if params is not None:
        default_params.update(params)

    categorical_feature_indices = list(range(len(CATEGORICAL_COLS)))

    return Pipeline([
        ("preprocess", preprocessor),
        ("model", HistGradientBoostingRegressor(
            **default_params,
            categorical_features=categorical_feature_indices,
        )),
    ])


## 10. HGB 기본 모델 학습

먼저 기본 파라미터로 학습한다. validation set과 test set에서 성능을 확인한 뒤, 튜닝 결과와 비교한다.


In [ ]:
hgb_base_model = build_hgb_pipeline()

hgb_base_model.fit(
    train[HGB_FEATURES],
    train["target_min"],
)

valid_pred_hgb_base = hgb_base_model.predict(valid[HGB_FEATURES])
test_pred_hgb_base = hgb_base_model.predict(test[HGB_FEATURES])

hgb_base_valid_metrics = evaluate_predictions(
    valid,
    valid_pred_hgb_base,
    long_wait_threshold,
)

hgb_base_test_metrics = evaluate_predictions(
    test,
    test_pred_hgb_base,
    long_wait_threshold,
)

hgb_base_results = pd.DataFrame([
    {"model": "HGB 기본", "split": "valid", **hgb_base_valid_metrics},
    {"model": "HGB 기본", "split": "test", **hgb_base_test_metrics},
])

display(
    hgb_base_results
    .drop(columns=["risk_Confusion_Matrix"])
    .round(4)
)

hgb_base_test_metrics["risk_Confusion_Matrix"]


## 11. HGB 하이퍼파라미터 튜닝

코랩이나 로컬에서 너무 오래 걸리지 않도록 작은 GridSearch부터 수행한다.

튜닝 기준은 validation set의 MAE를 1순위로 보고, 장시간 대기 위험 탐지 성능인 F1과 Recall을 보조로 본다.


In [ ]:
param_grid = {
    "learning_rate": [0.03, 0.05, 0.08],
    "max_iter": [200, 300],
    "max_leaf_nodes": [31, 63],
    "min_samples_leaf": [20, 50, 100],
    "l2_regularization": [0.0, 0.1],
}

grid = list(ParameterGrid(param_grid))

print(f"총 실험 조합 수: {len(grid)}")


In [ ]:
tuning_rows = []
best_params_mae = None
best_mae = np.inf

for i, params in enumerate(grid, start=1):
    print(f"[{i}/{len(grid)}] params: {params}")

    model = build_hgb_pipeline(params)

    model.fit(
        train[HGB_FEATURES],
        train["target_min"],
    )

    pred = model.predict(valid[HGB_FEATURES])

    reg = regression_metrics(valid["target_min"], pred)
    risk = risk_metrics(valid["target_min"], pred, long_wait_threshold)

    row = {
        "trial": i,
        **params,
        **reg,
        "risk_Accuracy": risk["Accuracy"],
        "risk_Precision": risk["Precision"],
        "risk_Recall": risk["Recall"],
        "risk_F1": risk["F1"],
    }

    tuning_rows.append(row)

    if reg["MAE"] < best_mae:
        best_mae = reg["MAE"]
        best_params_mae = params


tuning_results = (
    pd.DataFrame(tuning_rows)
    .sort_values(["MAE", "risk_F1"], ascending=[True, False])
    .reset_index(drop=True)
)

display(tuning_results.round(4))

best_params_mae


## 12. 최종 HGB 모델 학습 및 Test 평가

튜닝 결과 가장 좋은 파라미터를 사용해 train + validation을 합쳐 재학습한다.

최종 성능은 test set에서만 확인한다.


In [ ]:
train_valid = pd.concat([train, valid], ignore_index=True)

final_hgb_model = build_hgb_pipeline(best_params_mae)

final_hgb_model.fit(
    train_valid[HGB_FEATURES],
    train_valid["target_min"],
)

test_pred_hgb_final = final_hgb_model.predict(test[HGB_FEATURES])

final_hgb_test_metrics = evaluate_predictions(
    test,
    test_pred_hgb_final,
    long_wait_threshold,
)

final_hgb_test_metrics


In [ ]:
final_hgb_group_metrics = evaluate_by_group(
    test,
    test_pred_hgb_final,
    threshold_by_group,
)

display(final_hgb_group_metrics.round(4))


## 13. 예측 기준값 조정 실험

기본 장시간 대기 기준은 실제값 기준 train set의 90분위수다.

다만 예측값은 실제값보다 보수적으로 낮게 나올 수 있으므로, 장시간 대기 경고를 띄우는 예측 기준값은 별도로 조정할 수 있다.

예를 들어 실제 장시간 대기 기준이 약 94분이더라도, 예측값이 80분 이상이면 장시간 대기 위험으로 안내하는 방식이다.


In [ ]:
def prediction_threshold_sweep_fixed_actual(
    y_true,
    y_pred,
    actual_threshold,
    prediction_thresholds,
):
    rows = []
    actual = y_true >= actual_threshold

    for pred_threshold in prediction_thresholds:
        pred = y_pred >= pred_threshold

        rows.append({
            "actual_threshold": actual_threshold,
            "prediction_threshold": pred_threshold,
            "Accuracy": accuracy_score(actual, pred),
            "Precision": precision_score(actual, pred, zero_division=0),
            "Recall": recall_score(actual, pred, zero_division=0),
            "F1": f1_score(actual, pred, zero_division=0),
            "Confusion_Matrix": confusion_matrix(actual, pred, labels=[False, True]).tolist(),
        })

    return pd.DataFrame(rows)


prediction_threshold_candidates = [
    70,
    75,
    80,
    85,
    90,
    float(long_wait_threshold),
]

threshold_sweep_results = prediction_threshold_sweep_fixed_actual(
    test["target_min"].to_numpy(),
    test_pred_hgb_final,
    actual_threshold=long_wait_threshold,
    prediction_thresholds=prediction_threshold_candidates,
)

display(threshold_sweep_results.round(4))


## 14. RandomForest 최종 모델과 비교

06번 노트북의 RandomForest 최종 GridSearch 모델 결과와 HGB 결과를 비교한다.

RandomForest 최종 모델의 test 성능은 다음과 같았다.

```text
MAE: 15.0556
Median AE: 9.7643
RMSE: 23.5134
R²: 0.5734
장시간 대기 Precision: 0.7869
장시간 대기 Recall: 0.4206
장시간 대기 F1: 0.5482
```

HGB 모델은 이 결과와 비교하여 사용할지 판단한다.


In [ ]:
rf_final_reference = {
    "model": "RF GridSearch 최종",
    "MAE": 15.0556,
    "Median_AE": 9.7643,
    "RMSE": 23.5134,
    "R2": 0.5734,
    "risk_threshold": float(long_wait_threshold),
    "risk_Accuracy": 0.9318,
    "risk_Precision": 0.7869,
    "risk_Recall": 0.4206,
    "risk_F1": 0.5482,
}

hgb_final_row = {
    "model": "HGB 최종",
    **{
        k: v for k, v in final_hgb_test_metrics.items()
        if k != "risk_Confusion_Matrix"
    },
}

final_model_comparison = pd.DataFrame([
    rf_final_reference,
    hgb_final_row,
])

display(final_model_comparison.round(4))


## 15. 결과 저장

최종 HGB 모델, 장시간 대기율 proxy 테이블, 튜닝 결과, test 성능표를 저장한다.


In [ ]:
joblib.dump(
    final_hgb_model,
    MODEL_DIR / "hgb_wait_time_model.joblib",
)

joblib.dump(
    long_wait_rate_tables,
    MODEL_DIR / "hgb_long_wait_rate_tables.joblib",
)

pd.DataFrame([final_hgb_test_metrics]).to_csv(
    REPORT_DIR / "hgb_final_test_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

final_hgb_group_metrics.to_csv(
    REPORT_DIR / "hgb_final_test_metrics_by_group.csv",
    index=False,
    encoding="utf-8-sig",
)

tuning_results.to_csv(
    REPORT_DIR / "hgb_tuning_results.csv",
    index=False,
    encoding="utf-8-sig",
)

final_model_comparison.to_csv(
    REPORT_DIR / "hgb_vs_rf_final_comparison.csv",
    index=False,
    encoding="utf-8-sig",
)

metadata = {
    "model_name": "HistGradientBoostingRegressor",
    "target": "접수→승차 대기시간",
    "features": HGB_FEATURES,
    "categorical_cols": CATEGORICAL_COLS,
    "numeric_cols": NUMERIC_COLS,
    "long_wait_threshold": float(long_wait_threshold),
    "threshold_by_group": {k: float(v) for k, v in threshold_by_group.items()},
    "best_params_mae": best_params_mae,
}

with open(MODEL_DIR / "hgb_wait_time_model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

MODEL_DIR, REPORT_DIR


## 16. 해석 방향

HGB 모델 결과는 RandomForest 최종 모델과 비교한다.

비교할 핵심 지표는 다음과 같다.

```text
MAE
RMSE
R²
Recall
F1
```

해석 기준:

- MAE가 낮으면 평균적인 예상 대기시간 안내가 좋아진다.
- RMSE가 낮으면 장시간 대기처럼 큰 오차를 덜 낸다.
- R²가 높으면 전체 대기시간 변동을 더 잘 설명한다.
- Recall이 높으면 실제 장시간 대기 건을 더 많이 잡는다.
- F1이 높으면 장시간 대기 경고의 Precision과 Recall 균형이 좋아진다.

네비게이터 서비스에서는 MAE만 낮은 모델보다 장시간 대기 Recall과 F1이 함께 개선되는 모델이 더 적합하다.
